# Sistema de Machine Learning para Detecção de Anomalias IoT

## 📋 Visão Geral
Este notebook implementa um sistema completo de Machine Learning para detecção de anomalias em dados coletados por sensores ESP32 em um ambiente IoT.

## 🎯 Problema Escolhido
**Classificação de Anomalias em Leituras de Sensores IoT**

### Descrição do Problema:
- **Objetivo**: Detectar automaticamente anomalias em leituras de sensores ESP32
- **Tipo**: Classificação binária (Normal vs Anomalia)
- **Aplicação**: Sistema de monitoramento IoT para alertas preventivos
- **Impacto**: Reduzir falsos positivos e melhorar confiabilidade do sistema

### Justificativa:
1. **Prevenção de Falhas**: Detectar problemas antes que causem danos
2. **Otimização de Manutenção**: Alertas precisos reduzem intervenções desnecessárias
3. **Melhoria de Confiabilidade**: Sistema mais robusto e confiável
4. **Redução de Custos**: Menos paradas não programadas e manutenção preventiva

## 📊 Dataset Utilizado
**Dados Sintéticos Baseados em Parâmetros Reais**

### Características:
- **Tamanho**: 5.000 amostras
- **Features**: 15 variáveis (temperatura, umidade, pressão, vibração, nível, luminosidade, etc.)
- **Distribuição**: 70% Normal, 20% Alerta, 10% Falha
- **Anomalias**: 5% de anomalias extremas adicionais
- **Fonte**: Baseado nos dados reais coletados pelos sensores ESP32 do projeto

### Features Principais:
1. **Temperatura** (°C): -40 a 80°C
2. **Umidade** (%): 0 a 100%
3. **Pressão** (bar): 0 a 2 bar
4. **Vibração** (g): Magnitude da vibração triaxial
5. **Nível** (cm): 0 a 200 cm
6. **Luminosidade** (lux): 0 a 1023
7. **Movimento** (boolean): Detecção de movimento
8. **CO2** (ppm): 0 a 5000 ppm
9. **Ruído** (dB): 30 a 120 dB
10. **Features Derivadas**: Correlações e relações entre variáveis


In [ ]:
# Importar bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, 
                           precision_score, recall_score, f1_score, roc_auc_score, 
                           roc_curve, precision_recall_curve, average_precision_score)
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
import joblib
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo dos gráficos
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Bibliotecas importadas com sucesso!")
print("📊 Sistema de ML para Detecção de Anomalias IoT - Pronto para uso!")


## 🔧 1. Geração de Dados Sintéticos

Vamos gerar dados sintéticos baseados em parâmetros reais de sensores IoT para treinar nosso modelo.


In [ ]:
def gerar_dados_sinteticos(n_samples=5000, random_state=42):
    """
    Gera dados sintéticos baseados em parâmetros reais de sensores IoT
    """
    print("🔄 Gerando dados sintéticos para treinamento...")
    
    np.random.seed(random_state)
    data = []
    
    for i in range(n_samples):
        # Simular diferentes modos de operação
        mode = np.random.choice([0, 1, 2], p=[0.7, 0.2, 0.1])  # Normal, Alerta, Falha
        
        if mode == 0:  # Normal
            temperature = np.random.normal(25.0, 2.0)
            humidity = np.random.normal(60.0, 5.0)
            pressure = np.random.normal(1.013, 0.01)
            vibration_x = np.random.normal(0.0, 0.1)
            vibration_y = np.random.normal(0.0, 0.1)
            vibration_z = np.random.normal(0.0, 0.1)
            level = np.random.normal(100.0, 10.0)
            luminosity = np.random.normal(500.0, 100.0)
            movement = np.random.choice([0, 1], p=[0.9, 0.1])
            co2 = np.random.normal(400.0, 50.0)
            noise = np.random.normal(45.0, 5.0)
            
        elif mode == 1:  # Alerta
            temperature = np.random.normal(28.0, 3.0)
            humidity = np.random.normal(70.0, 8.0)
            pressure = np.random.normal(1.020, 0.02)
            vibration_x = np.random.normal(0.3, 0.2)
            vibration_y = np.random.normal(0.2, 0.15)
            vibration_z = np.random.normal(0.1, 0.1)
            level = np.random.normal(120.0, 15.0)
            luminosity = np.random.normal(600.0, 150.0)
            movement = np.random.choice([0, 1], p=[0.7, 0.3])
            co2 = np.random.normal(800.0, 100.0)
            noise = np.random.normal(55.0, 8.0)
            
        else:  # Falha
            temperature = np.random.normal(35.0, 5.0)
            humidity = np.random.normal(85.0, 10.0)
            pressure = np.random.normal(1.030, 0.05)
            vibration_x = np.random.normal(1.0, 0.5)
            vibration_y = np.random.normal(0.8, 0.4)
            vibration_z = np.random.normal(0.6, 0.3)
            level = np.random.normal(150.0, 25.0)
            luminosity = np.random.normal(800.0, 200.0)
            movement = np.random.choice([0, 1], p=[0.5, 0.5])
            co2 = np.random.normal(1500.0, 300.0)
            noise = np.random.normal(75.0, 15.0)
        
        # Adicionar anomalias ocasionais (5% de chance)
        if np.random.random() < 0.05:
            temperature += np.random.normal(0, 10)
            humidity += np.random.normal(0, 20)
            pressure += np.random.normal(0, 0.1)
            vibration_x += np.random.normal(0, 0.5)
            vibration_y += np.random.normal(0, 0.5)
            vibration_z += np.random.normal(0, 0.5)
            level += np.random.normal(0, 30)
            luminosity += np.random.normal(0, 300)
            co2 += np.random.normal(0, 500)
            noise += np.random.normal(0, 20)
            mode = 2  # Marcar como falha
        
        # Calcular magnitude da vibração
        vibration_mag = np.sqrt(vibration_x**2 + vibration_y**2 + vibration_z**2)
        
        # Features derivadas
        temp_humidity_ratio = temperature / (humidity + 1)
        pressure_vibration = pressure * vibration_mag
        level_luminosity = level * luminosity / 1000
        temp_pressure_ratio = temperature / (pressure + 0.1)
        humidity_level_ratio = humidity / (level + 1)
        co2_noise_ratio = co2 / (noise + 1)
        
        data.append({
            'temperature': max(-40, min(80, round(temperature, 2))),
            'humidity': max(0, min(100, round(humidity, 2))),
            'pressure': max(0, min(2, round(pressure, 3))),
            'vibration_x': round(vibration_x, 3),
            'vibration_y': round(vibration_y, 3),
            'vibration_z': round(vibration_z, 3),
            'vibration_mag': round(vibration_mag, 3),
            'level': max(0, min(200, round(level, 1))),
            'luminosity': max(0, min(1023, round(luminosity, 0))),
            'movement': int(movement),
            'co2': max(0, min(5000, round(co2, 0))),
            'noise': max(30, min(120, round(noise, 1))),
            'temp_humidity_ratio': round(temp_humidity_ratio, 4),
            'pressure_vibration': round(pressure_vibration, 4),
            'level_luminosity': round(level_luminosity, 2),
            'temp_pressure_ratio': round(temp_pressure_ratio, 2),
            'humidity_level_ratio': round(humidity_level_ratio, 4),
            'co2_noise_ratio': round(co2_noise_ratio, 2),
            'anomaly': 1 if mode == 2 else 0,
            'mode': mode
        })
    
    df = pd.DataFrame(data)
    print(f"✅ Dados sintéticos gerados: {len(df)} amostras")
    print(f"   • Normal: {len(df[df['anomaly'] == 0])} amostras")
    print(f"   • Anomalia: {len(df[df['anomaly'] == 1])} amostras")
    
    return df

# Gerar dados
df = gerar_dados_sinteticos(n_samples=5000)
print(f"\n📊 Primeiras 5 linhas do dataset:")
df.head()


## 📊 2. Análise Exploratória dos Dados

Vamos analisar a distribuição dos dados e entender as características do dataset.


In [ ]:
# Análise exploratória dos dados
print("📊 INFORMAÇÕES BÁSICAS DO DATASET")
print("="*50)
print(f"Shape: {df.shape}")
print(f"Colunas: {list(df.columns)}")
print(f"Tipos de dados:")
print(df.dtypes)

print(f"\n📈 ESTATÍSTICAS DESCRITIVAS")
print("="*50)
print(df.describe())

print(f"\n🔍 DISTRIBUIÇÃO DAS CLASSES")
print("="*50)
print(f"Normal: {len(df[df['anomaly'] == 0])} amostras ({len(df[df['anomaly'] == 0])/len(df):.1%})")
print(f"Anomalia: {len(df[df['anomaly'] == 1])} amostras ({len(df[df['anomaly'] == 1])/len(df):.1%})")

# Verificar valores nulos
print(f"\n❌ VALORES NULOS")
print("="*50)
print(df.isnull().sum())

# Verificar valores infinitos
print(f"\n♾️ VALORES INFINITOS")
print("="*50)
print(df.isin([np.inf, -np.inf]).sum())


In [ ]:
# Visualizações da distribuição dos dados
fig, axes = plt.subplots(3, 3, figsize=(18, 15))
fig.suptitle('Distribuição dos Dados por Classe', fontsize=16, fontweight='bold')

# Features principais para visualização
features_viz = ['temperature', 'humidity', 'pressure', 'vibration_mag', 
                'level', 'luminosity', 'co2', 'noise', 'temp_humidity_ratio']

for i, feature in enumerate(features_viz):
    row = i // 3
    col = i % 3
    
    # Histograma por classe
    normal_data = df[df['anomaly'] == 0][feature]
    anomaly_data = df[df['anomaly'] == 1][feature]
    
    axes[row, col].hist(normal_data, bins=30, alpha=0.7, label='Normal', color='blue', density=True)
    axes[row, col].hist(anomaly_data, bins=30, alpha=0.7, label='Anomalia', color='red', density=True)
    axes[row, col].set_title(f'Distribuição de {feature}', fontweight='bold')
    axes[row, col].set_xlabel(feature)
    axes[row, col].set_ylabel('Densidade')
    axes[row, col].legend()
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Matriz de correlação
plt.figure(figsize=(15, 12))
correlation_matrix = df[features_viz + ['anomaly']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlação das Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 🤖 3. Preparação dos Dados e Treinamento do Modelo

Agora vamos preparar os dados e treinar nosso modelo de detecção de anomalias.


In [ ]:
# Preparar dados para treinamento
def preparar_dados(df):
    """
    Prepara os dados para treinamento
    """
    print("🔧 Preparando dados para treinamento...")
    
    # Selecionar features
    feature_columns = [
        'temperature', 'humidity', 'pressure', 'vibration_mag',
        'level', 'luminosity', 'movement', 'co2', 'noise',
        'temp_humidity_ratio', 'pressure_vibration', 'level_luminosity',
        'temp_pressure_ratio', 'humidity_level_ratio', 'co2_noise_ratio'
    ]
    
    X = df[feature_columns].copy()
    y = df['anomaly'].copy()
    
    # Tratar valores infinitos e nulos
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())
    
    print(f"   • Features: {X.shape[1]}")
    print(f"   • Amostras: {X.shape[0]}")
    print(f"   • Distribuição: {y.value_counts().to_dict()}")
    
    return X, y, feature_columns

# Preparar dados
X, y, feature_columns = preparar_dados(df)

# Dividir dados em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 DIVISÃO DOS DADOS")
print("="*50)
print(f"Treino: {X_train.shape[0]} amostras ({X_train.shape[0]/len(X):.1%})")
print(f"Teste: {X_test.shape[0]} amostras ({X_test.shape[0]/len(X):.1%})")

# Normalizar dados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Dados preparados e normalizados!")


In [ ]:
# Treinar modelo Random Forest
print("🤖 Treinando modelo Random Forest...")

# Modelo Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    class_weight='balanced'
)

# Treinar modelo
rf_model.fit(X_train_scaled, y_train)

# Fazer predições
y_pred = rf_model.predict(X_test_scaled)
y_pred_proba = rf_model.predict_proba(X_test_scaled)[:, 1]

# Calcular métricas
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"✅ Modelo Random Forest treinado!")
print(f"   • Accuracy: {accuracy:.4f}")
print(f"   • Precision: {precision:.4f}")
print(f"   • Recall: {recall:.4f}")
print(f"   • F1-Score: {f1:.4f}")
print(f"   • AUC: {auc:.4f}")

# Treinar Isolation Forest para comparação
print(f"\n🔍 Treinando Isolation Forest...")
isolation_forest = IsolationForest(
    contamination=0.1,
    random_state=42
)
isolation_forest.fit(X_train_scaled)

# Predições do Isolation Forest
y_pred_iso = isolation_forest.predict(X_test_scaled)
y_pred_iso = (y_pred_iso == -1).astype(int)  # Converter -1/1 para 0/1

# Métricas do Isolation Forest
accuracy_iso = accuracy_score(y_test, y_pred_iso)
precision_iso = precision_score(y_test, y_pred_iso)
recall_iso = recall_score(y_test, y_pred_iso)
f1_iso = f1_score(y_test, y_pred_iso)

print(f"✅ Isolation Forest treinado!")
print(f"   • Accuracy: {accuracy_iso:.4f}")
print(f"   • Precision: {precision_iso:.4f}")
print(f"   • Recall: {recall_iso:.4f}")
print(f"   • F1-Score: {f1_iso:.4f}")

# Comparação dos modelos
print(f"\n📊 COMPARAÇÃO DOS MODELOS")
print("="*50)
print(f"{'Métrica':<15} {'Random Forest':<15} {'Isolation Forest':<15}")
print("-" * 50)
print(f"{'Accuracy':<15} {accuracy:<15.4f} {accuracy_iso:<15.4f}")
print(f"{'Precision':<15} {precision:<15.4f} {precision_iso:<15.4f}")
print(f"{'Recall':<15} {recall:<15.4f} {recall_iso:<15.4f}")
print(f"{'F1-Score':<15} {f1:<15.4f} {f1_iso:<15.4f}")


## 📊 4. Visualização dos Resultados

Vamos visualizar os resultados do modelo e analisar sua performance.


In [ ]:
# Visualizações dos resultados
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Resultados do Modelo de Detecção de Anomalias', fontsize=16, fontweight='bold')

# 1. Matriz de confusão
ax1 = axes[0, 0]
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
           xticklabels=['Normal', 'Anomalia'],
           yticklabels=['Normal', 'Anomalia'], ax=ax1)
ax1.set_title('Matriz de Confusão - Random Forest', fontweight='bold')
ax1.set_xlabel('Predição')
ax1.set_ylabel('Valor Real')

# 2. Curva ROC
ax2 = axes[0, 1]
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
auc_score = roc_auc_score(y_test, y_pred_proba)
ax2.plot(fpr, tpr, color='darkorange', lw=2, 
        label=f'ROC Curve (AUC = {auc_score:.4f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Taxa de Falsos Positivos')
ax2.set_ylabel('Taxa de Verdadeiros Positivos')
ax2.set_title('Curva ROC', fontweight='bold')
ax2.legend(loc="lower right")
ax2.grid(True, alpha=0.3)

# 3. Curva Precision-Recall
ax3 = axes[0, 2]
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_pred_proba)
ap_score = average_precision_score(y_test, y_pred_proba)
ax3.plot(recall_curve, precision_curve, color='darkorange', lw=2,
        label=f'PR Curve (AP = {ap_score:.4f})')
ax3.set_xlim([0.0, 1.0])
ax3.set_ylim([0.0, 1.05])
ax3.set_xlabel('Recall')
ax3.set_ylabel('Precision')
ax3.set_title('Curva Precision-Recall', fontweight='bold')
ax3.legend(loc="lower left")
ax3.grid(True, alpha=0.3)

# 4. Importância das features
ax4 = axes[1, 0]
feature_importance = rf_model.feature_importances_
indices = np.argsort(feature_importance)[::-1]
bars = ax4.bar(range(len(feature_importance)), feature_importance[indices])
ax4.set_title('Importância das Features', fontweight='bold')
ax4.set_xlabel('Features')
ax4.set_ylabel('Importância')
ax4.set_xticks(range(len(feature_columns)))
ax4.set_xticklabels([feature_columns[i] for i in indices], rotation=45, ha='right')

# Colorir barras
colors = plt.cm.viridis(np.linspace(0, 1, len(bars)))
for bar, color in zip(bars, colors):
    bar.set_color(color)

ax4.grid(True, alpha=0.3, axis='y')

# 5. Distribuição das predições
ax5 = axes[1, 1]
ax5.hist(y_pred_proba[y_test == 0], bins=30, alpha=0.7, label='Normal', color='blue')
ax5.hist(y_pred_proba[y_test == 1], bins=30, alpha=0.7, label='Anomalia', color='red')
ax5.set_xlabel('Probabilidade de Anomalia')
ax5.set_ylabel('Frequência')
ax5.set_title('Distribuição das Predições', fontweight='bold')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. Métricas de performance
ax6 = axes[1, 2]
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']
values = [accuracy, precision, recall, f1, auc]
bars = ax6.bar(metrics, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
ax6.set_title('Métricas de Performance', fontweight='bold')
ax6.set_ylabel('Score')
ax6.set_xticklabels(metrics, rotation=45)

# Adicionar valores nas barras
for bar, value in zip(bars, values):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{value:.3f}', ha='center', va='bottom')

ax6.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## 🔍 5. Validação Cruzada e Otimização

Vamos validar nosso modelo e otimizar os hiperparâmetros.


In [ ]:
# Validação cruzada
print("🔄 Executando validação cruzada...")

cv_scores = cross_val_score(
    rf_model, X_train_scaled, y_train, cv=5, scoring='roc_auc'
)

print(f"✅ Validação cruzada (5 folds):")
print(f"   • Scores: {cv_scores}")
print(f"   • Média: {cv_scores.mean():.4f}")
print(f"   • Desvio padrão: {cv_scores.std():.4f}")
print(f"   • Intervalo: {cv_scores.mean():.4f} ± {cv_scores.std() * 2:.4f}")

# Otimização de hiperparâmetros
print(f"\n🔍 Otimizando hiperparâmetros...")

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced'),
    param_grid, cv=3, scoring='roc_auc', n_jobs=-1
)

grid_search.fit(X_train_scaled, y_train)

print(f"✅ Melhores parâmetros: {grid_search.best_params_}")
print(f"   • Melhor score: {grid_search.best_score_:.4f}")

# Treinar modelo otimizado
best_model = grid_search.best_estimator_
best_model.fit(X_train_scaled, y_train)

# Predições do modelo otimizado
y_pred_optimized = best_model.predict(X_test_scaled)
y_pred_proba_optimized = best_model.predict_proba(X_test_scaled)[:, 1]

# Métricas do modelo otimizado
accuracy_opt = accuracy_score(y_test, y_pred_optimized)
precision_opt = precision_score(y_test, y_pred_optimized)
recall_opt = recall_score(y_test, y_pred_optimized)
f1_opt = f1_score(y_test, y_pred_optimized)
auc_opt = roc_auc_score(y_test, y_pred_proba_optimized)

print(f"\n📊 COMPARAÇÃO: MODELO ORIGINAL vs OTIMIZADO")
print("="*60)
print(f"{'Métrica':<15} {'Original':<15} {'Otimizado':<15} {'Melhoria':<15}")
print("-" * 60)
print(f"{'Accuracy':<15} {accuracy:<15.4f} {accuracy_opt:<15.4f} {accuracy_opt-accuracy:<15.4f}")
print(f"{'Precision':<15} {precision:<15.4f} {precision_opt:<15.4f} {precision_opt-precision:<15.4f}")
print(f"{'Recall':<15} {recall:<15.4f} {recall_opt:<15.4f} {recall_opt-recall:<15.4f}")
print(f"{'F1-Score':<15} {f1:<15.4f} {f1_opt:<15.4f} {f1_opt-f1:<15.4f}")
print(f"{'AUC':<15} {auc:<15.4f} {auc_opt:<15.4f} {auc_opt-auc:<15.4f}")

# Usar modelo otimizado como final
rf_model = best_model
y_pred = y_pred_optimized
y_pred_proba = y_pred_proba_optimized
accuracy = accuracy_opt
precision = precision_opt
recall = recall_opt
f1 = f1_opt
auc = auc_opt


## 💾 6. Salvamento do Modelo e Relatório Final

Vamos salvar o modelo treinado e gerar um relatório final.


In [ ]:
# Salvar modelo treinado
print("💾 Salvando modelo treinado...")

model_data = {
    'model': rf_model,
    'scaler': scaler,
    'feature_columns': feature_columns,
    'isolation_forest': isolation_forest
}

joblib.dump(model_data, 'modelo_anomalia_iot_completo.pkl')
print("✅ Modelo salvo como: modelo_anomalia_iot_completo.pkl")

# Gerar relatório final
print(f"\n📋 RELATÓRIO FINAL - MODELO DE DETECÇÃO DE ANOMALIAS")
print("="*70)

print(f"\n📊 MÉTRICAS PRINCIPAIS:")
print(f"   • Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   • Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"   • Recall: {recall:.4f} ({recall*100:.2f}%)")
print(f"   • F1-Score: {f1:.4f} ({f1*100:.2f}%)")
print(f"   • AUC: {auc:.4f} ({auc*100:.2f}%)")

print(f"\n📈 VALIDAÇÃO CRUZADA:")
print(f"   • Média: {cv_scores.mean():.4f}")
print(f"   • Desvio padrão: {cv_scores.std():.4f}")
print(f"   • Intervalo de confiança: {cv_scores.mean():.4f} ± {cv_scores.std() * 2:.4f}")

print(f"\n🔍 TOP 10 FEATURES MAIS IMPORTANTES:")
feature_importance = rf_model.feature_importances_
feature_importance_df = pd.DataFrame({
    'feature': feature_columns,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

for i, row in feature_importance_df.head(10).iterrows():
    print(f"   {row['feature']}: {row['importance']:.4f}")

print(f"\n📊 DISTRIBUIÇÃO DAS PREDIÇÕES:")
print(f"   • Total de amostras: {len(y_test)}")
print(f"   • Anomalias reais: {sum(y_test)}")
print(f"   • Anomalias preditas: {sum(y_pred)}")
print(f"   • Verdadeiros positivos: {sum((y_test == 1) & (y_pred == 1))}")
print(f"   • Falsos positivos: {sum((y_test == 0) & (y_pred == 1))}")
print(f"   • Verdadeiros negativos: {sum((y_test == 0) & (y_pred == 0))}")
print(f"   • Falsos negativos: {sum((y_test == 1) & (y_pred == 0))}")

print(f"\n✅ MODELO PRONTO PARA USO EM PRODUÇÃO!")
print("="*70)

# Exemplo de uso do modelo
print(f"\n🔧 EXEMPLO DE USO DO MODELO:")
print("="*50)

# Criar dados de exemplo
exemplo_dados = pd.DataFrame({
    'temperature': [25.5, 35.0, 20.0],
    'humidity': [60.0, 85.0, 45.0],
    'pressure': [1.013, 1.030, 1.010],
    'vibration_mag': [0.1, 1.2, 0.05],
    'level': [100.0, 150.0, 80.0],
    'luminosity': [500.0, 800.0, 300.0],
    'movement': [0, 1, 0],
    'co2': [400.0, 1500.0, 350.0],
    'noise': [45.0, 75.0, 40.0],
    'temp_humidity_ratio': [0.425, 0.412, 0.444],
    'pressure_vibration': [0.101, 1.236, 0.051],
    'level_luminosity': [50.0, 120.0, 24.0],
    'temp_pressure_ratio': [25.2, 34.0, 19.8],
    'humidity_level_ratio': [0.6, 0.567, 0.563],
    'co2_noise_ratio': [8.89, 20.0, 8.75]
})

# Fazer predições
X_exemplo = exemplo_dados[feature_columns]
X_exemplo_scaled = scaler.transform(X_exemplo)
predicoes_exemplo = rf_model.predict(X_exemplo_scaled)
probabilidades_exemplo = rf_model.predict_proba(X_exemplo_scaled)[:, 1]

print("Dados de exemplo:")
print(exemplo_dados[['temperature', 'humidity', 'pressure', 'vibration_mag', 'co2']])
print(f"\nPredições: {predicoes_exemplo}")
print(f"Probabilidades: {probabilidades_exemplo}")

print(f"\n🎯 SISTEMA COMPLETO E PRONTO PARA USO!")
print("   • Modelo treinado e validado")
print("   • Visualizações geradas")
print("   • Relatório detalhado criado")
print("   • Modelo salvo para uso em produção")
